# Vectors, dot product, cosine similarity, norms

*0.2 Math / ML basics · run **Setup** first*

## Setup

Settings, a configured client, an `embed()` helper and numpy. Every cell below uses them.

In [1]:
"""Shared setup: typed settings, a configured client, numpy, and a print helper."""

import json

import numpy as np
from dotenv import find_dotenv
from openai import OpenAI
from pydantic import Field, SecretStr
from pydantic_settings import BaseSettings, SettingsConfigDict

np.set_printoptions(precision=4, suppress=True)


class Settings(BaseSettings):
    model_config = SettingsConfigDict(env_file=find_dotenv(), extra="ignore")

    openai_api_key: SecretStr
    openai_model: str = "gpt-4o-mini"
    embedding_model: str = "text-embedding-3-small"
    request_timeout_seconds: float = Field(default=30, gt=0)
    max_retries: int = Field(default=2, ge=0, le=5)


settings = Settings()
client = OpenAI(
    api_key=settings.openai_api_key.get_secret_value(),
    timeout=settings.request_timeout_seconds,
    max_retries=settings.max_retries,
)


def embed(texts: list[str]) -> np.ndarray:
    """Embed texts with the configured model; returns one row per text."""
    response = client.embeddings.create(model=settings.embedding_model, input=texts)
    rows = []
    for item in response.data:
        rows.append(item.embedding)
    return np.array(rows)


def show(title: str, value) -> None:
    print(title)
    print(json.dumps(value, indent=2, ensure_ascii=False, default=str))


print("chat model:", settings.openai_model, "| embedding model:", settings.embedding_model)

chat model: gpt-4o-mini | embedding model: text-embedding-3-small


### Vectors

> **Problem.** A search box has to find "kitten" when the user types "cat". Text cannot be compared for meaning; only numbers can. Embedding models turn each text into a list of 1,536 numbers — a vector — and from then on every question about meaning is a question about numbers.

**Idea.** A vector is an ordered list of numbers; an embedding is a vector whose numbers encode meaning.

**Use when** you need to compare, search or cluster text by meaning.  
**Not when** exact matching is what you want (ids, codes, keywords) — use a plain index.

```
"cat"     ──embed──▶  [ 0.021, -0.013,  0.048, …, 0.007 ]   1,536 numbers
"kitten"  ──embed──▶  [ 0.019, -0.011,  0.051, …, 0.006 ]   close to cat's
"car"     ──embed──▶  [-0.032,  0.044, -0.008, …, 0.019 ]   far from both
```

**How it works.**
1. `embed(words)` sends three strings to the embedding model and gets back one row of 1,536 numbers per string — a matrix of shape (3, 1536).
2. Each row is a point in a 1,536-dimensional space; texts with similar meaning land near each other.
3. Vector arithmetic is element-wise: `cat + kitten` adds position by position, `2 * cat` doubles every number.
4. Everything downstream — similarity, search, clustering — is numpy operations on these rows.

| | what happens | result |
|:--|:--|:--|
| ✗ compare strings | `"cat" == "kitten"` | False — no notion of meaning |
| ✓ compare vectors | `embed("cat")` vs `embed("kitten")` | numbers that can be scored for closeness |

**Production code and its real output**

In [2]:
# Vectors — embeddings are vectors; numpy is the production tool for working with them.
words = ["cat", "kitten", "car"]
vectors = embed(words)
print("shape:", vectors.shape, "(3 words × 1536 dimensions)")
print("first 6 values of 'cat':", vectors[0][:6])

# Vector arithmetic is element-wise.
cat, kitten, car = vectors[0], vectors[1], vectors[2]
print("cat + kitten (first 4):", (cat + kitten)[:4])
print("2 * cat      (first 4):", (2 * cat)[:4])
assert vectors.shape == (3, 1536)

shape: (3, 1536) (3 words × 1536 dimensions)
first 6 values of 'cat': [ 0.0256 -0.0234 -0.0161  0.0394  0.021  -0.0263]
cat + kitten (first 4): [ 0.0191 -0.0759 -0.0322  0.026 ]
2 * cat      (first 4): [ 0.0511 -0.0468 -0.0321  0.0788]


**What the output shows.** Three words became a 3 × 1536 matrix; the first six numbers of each row are shown. The arithmetic lines confirm rows are ordinary numpy arrays.

**In practice**
- **dimension is a cost** — 1,536 floats × 4 bytes ≈ 6 KB per text; 10 million texts ≈ 60 GB. Smaller models or `dimensions=256` cut storage 6×.
- **one model per index** — vectors from different embedding models live in different spaces; never mix them in one index.
- **batch the calls** — the API accepts up to 2,048 inputs per request; embedding one text at a time is 100× slower and costs the same.
- **float32 is enough** — store as float32 (or int8-quantised); float64 doubles storage for no gain.

**Alternatives** — sparse vectors (TF-IDF, BM25) for keyword-style matching · hybrid: both, fused

**Terms** — *vector*: an ordered list of numbers · *embedding*: a vector that encodes the meaning of a text · *dimension*: how many numbers the vector has


### dot product

> **Problem.** You have 10 million document vectors and one query vector. The search must return the closest documents in milliseconds. Whatever "closest" means, it has to be a single fast operation the hardware is good at.

**Idea.** Multiply two vectors element by element and add up — one number that grows when they point the same way.

**Use when** scoring a query against many vectors at once (search, reranking, attention).  
**Not when** vectors have different lengths — a long vector wins regardless of direction (use cosine).

```
cat     [ 0.02  -0.01   0.05 ]
kitten  [ 0.02  -0.01   0.05 ]
         ─────  ─────  ─────
          ×      ×      ×      →  0.0004 + 0.0001 + 0.0025 + … = 0.55   (high: same direction)
cat · car                                                       = 0.13   (low)
```

**How it works.**
1. `cat @ kitten` multiplies the 1,536 pairs of numbers and sums them — numpy's `@` is the dot product.
2. A high value means the two vectors point the same way; near zero means unrelated.
3. `vectors @ cat` scores every row against the query in one matrix-vector product — this is what a vector database does per query.
4. `np.argmax(scores)` picks the best match; a real index does this over millions of rows with approximate search.

| | what happens | result |
|:--|:--|:--|
| ✓ | `cat @ kitten` | high score — related |
| ✓ | `cat @ car` | low score — unrelated |
| ✓ | `matrix @ cat` | all scores in one operation |

**Production code and its real output**

In [3]:
# Dot product — the score behind similarity search: multiply element-wise, sum. numpy's @.
words = ["cat", "kitten", "car"]
vectors = embed(words)
cat, kitten, car = vectors[0], vectors[1], vectors[2]

print("cat · kitten:", round(float(cat @ kitten), 4))
print("cat · car:   ", round(float(cat @ car), 4))

# One query against many rows at once: a matrix-vector product.
scores = vectors @ cat
print("all rows · cat:", scores, "→ best match index", int(np.argmax(scores)))
assert float(cat @ kitten) > float(cat @ car)

cat · kitten: 0.5696
cat · car:    0.516
all rows · cat: [0.9999 0.5696 0.516 ] → best match index 0


**What the output shows.** `cat · kitten` scored well above `cat · car`, and the matrix product returned all three scores at once with `cat` itself as the best match.

**In practice**
- **hardware** — dot products are what GPUs and SIMD CPUs do fastest; every retrieval and attention system is built on them.
- **normalise first** — on unit-length vectors the dot product *is* cosine similarity — vector databases normalise at insert so queries can use the cheaper operation.
- **batch queries** — `queries @ vectors.T` scores 100 queries against 1 million rows in one call; loops over queries are 100× slower.
- **approximate search** — at millions of rows exact dot products are too slow; HNSW/IVF indexes trade a little recall for 100× speed (layer 4).

**Alternatives** — cosine similarity (direction only) · euclidean distance (magnitude-aware) · learned rerankers for the top few

**Terms** — *dot product*: sum of element-wise products · *argmax*: the index of the largest value


### cosine similarity

> **Problem.** Two documents about the same topic score differently by dot product just because one is longer and its vector is larger. Ranking by dot product then favours long or "loud" documents over relevant ones.

**Idea.** Dot product divided by both lengths — only the angle between the vectors counts, never their size.

**Use when** ranking text by meaning; comparing vectors that may differ in scale.  
**Not when** magnitude carries information you want (rare for text embeddings).

```mermaid
flowchart LR
    A["cat · kitten"] --> D["÷ (|cat| × |kitten|)"] --> S["cosine 0.57 · direction only"]
```

**How it works.**
1. `cosine_similarity(vectors)` from sklearn computes every pair: dot product divided by the product of the two lengths.
2. The result is between −1 and 1; 1 means identical direction, 0 unrelated, negative opposite.
3. The matrix printed is symmetric: `cat` vs `kitten` is high, `car` vs `automobile` is high, cross pairs are low.
4. Scaling a vector by 10 leaves its cosine with itself at exactly 1 — length has been divided out.

| | what happens | result |
|:--|:--|:--|
| ✗ dot product | long document vector | scores high for being long |
| ✓ cosine | same vectors | scores by direction only — relevance |

**Production code and its real output**

In [4]:
# Cosine similarity — dot product divided by both lengths; only direction matters. sklearn does it.
from sklearn.metrics.pairwise import cosine_similarity

words = ["cat", "kitten", "car", "automobile"]
vectors = embed(words)
matrix = cosine_similarity(vectors)

header = f"{'':<11}"
for word in words:
    header += f"{word:>11}"
print(header)
for word, row in zip(words, matrix, strict=True):
    line = f"{word:<11}"
    for value in row:
        line += f"{value:>11.3f}"
    print(line)

# Scaling a vector changes the dot product but not the cosine.
scaled = 10 * vectors[0:1]
print("cosine(cat, 10×cat):", round(float(cosine_similarity(vectors[0:1], scaled)[0, 0]), 4))
assert matrix[0, 1] > matrix[0, 2] and matrix[2, 3] > matrix[1, 3]

                   cat     kitten        car automobile
cat              1.000      0.570      0.516      0.316
kitten           0.570      1.000      0.360      0.248
car              0.516      0.360      1.000      0.556
automobile       0.316      0.248      0.556      1.000
cosine(cat, 10×cat): 1.0


**What the output shows.** The 4 × 4 matrix groups the two animals and the two vehicles; `cosine(cat, 10 × cat)` is exactly 1.0, proving scale does not matter.

**In practice**
- **normalise at insert** — store unit-length vectors; then cosine is a plain dot product and the index can use the fast path.
- **thresholds are model-specific** — 0.8 means "very similar" for one model and "barely related" for another; calibrate on your own data.
- **not a probability** — cosine 0.9 is not 90% relevant; use it to rank, not to decide.
- **small differences matter** — with modern embeddings most scores sit between 0.2 and 0.6; the ranking is meaningful even when absolute values look close.

**Alternatives** — dot product on normalised vectors (identical, faster) · euclidean distance · cross-encoder rerankers for the final ordering

**Terms** — *cosine similarity*: dot product of the two vectors divided by their lengths · *unit vector*: a vector of length 1


### norms

> **Problem.** A vector database offers "cosine", "dot" and "euclidean" as distance options and the results differ. To pick one — and to know when they are the same — you need the notion of a vector's length.

**Idea.** A norm is a vector's length; dividing by it makes a unit vector, after which dot product, cosine and euclidean distance all agree.

**Use when** preparing vectors for storage or comparison; choosing a distance metric.  
**Not when** —.

```
v = [3, 4]        L2 norm = √(3² + 4²) = 5        unit vector = v / 5 = [0.6, 0.8]

for unit vectors:   dot = cosine          euclidean² = 2 − 2·cosine
```

**How it works.**
1. `np.linalg.norm(vectors, axis=1)` gives each row's L2 length; OpenAI embeddings come back at ≈1.0 but not exactly.
2. `normalize(vectors)` from sklearn divides each row by its length, giving exactly unit-length rows.
3. On unit vectors the dot product equals the cosine similarity — the cell prints one number for both.
4. The squared euclidean distance between unit vectors is `2 − 2·cosine`, so the three metrics rank identically.

| | what happens | result |
|:--|:--|:--|
| ✗ raw vectors | dot ≠ cosine ≠ euclidean ranking | three different answers |
| ✓ unit vectors | dot = cosine, euclidean monotone | one answer, cheapest operation |

**Production code and its real output**

In [5]:
# Norms — a vector's length. Unit-normalising vectors makes dot product equal cosine, which is
# why vector databases store normalised embeddings.
from sklearn.preprocessing import normalize

vectors = embed(["cat", "kitten"])
lengths = np.linalg.norm(vectors, axis=1)
print("L2 length as returned:", lengths, "(≈1 — OpenAI embeddings are nearly unit length)")

unit = normalize(vectors)  # exact unit length
print("after normalize():    ", np.linalg.norm(unit, axis=1))

cosine = float(unit[0] @ unit[1])
distance = float(np.linalg.norm(unit[0] - unit[1]))
print(f"dot product of unit vectors = cosine = {cosine:.4f}")
print(f"euclidean distance² = 2 − 2·cosine → {distance**2:.4f} vs {2 - 2 * cosine:.4f}")
assert abs(distance**2 - (2 - 2 * cosine)) < 1e-6

L2 length as returned: [0.9999 0.9998] (≈1 — OpenAI embeddings are nearly unit length)
after normalize():     [1. 1.]
dot product of unit vectors = cosine = 0.5698
euclidean distance² = 2 − 2·cosine → 0.8604 vs 0.8604


**What the output shows.** Lengths were ≈0.9998 before and exactly 1.0 after normalising; the dot product equalled the cosine and the distance identity held to six decimals.

**In practice**
- **normalise once** — at insert time, not per query; store the unit vector.
- **pick dot after normalising** — it is the cheapest metric and the index's fast path; cosine and euclidean give the same order.
- **do not assume unit length** — some models (and all fine-tuned ones) return arbitrary lengths; check `norm` once per model.
- **L1 and L∞** — rarely used for embeddings; L1 for sparse vectors, L∞ for bounding values.

**Alternatives** — euclidean distance on raw vectors when magnitude matters (image features, some audio embeddings)

**Terms** — *L2 norm*: the usual straight-line length · *normalise*: divide by the length so it becomes 1 · *metric*: the rule for measuring distance
